In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

from scipy.stats import median_abs_deviation

In [3]:
sc.settings.verbosity = 0             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor='white')

scanpy==1.10.4 anndata==0.11.3 umap==0.5.7 numpy==1.26.4 scipy==1.15.1 pandas==2.2.3 scikit-learn==1.6.1 statsmodels==0.14.4 igraph==0.11.8 pynndescent==0.5.13


In [5]:
samples = [
    "PBMC_10x", "PBMC_10x_2", "PBMC_10x_3", "PBMC_indrops", "PBMC_indrops_2", 
    "brain", "brain_2", "eye", "eye_2", "eye_3", "lung_2", "lung_5", "lung_7", "lung_8"
]

In [4]:
samples = ["PBMC_10x_2"]

In [7]:
sample = "PBMC_10x_2"

In [24]:
sample = "PBMC_10x_2"
adata_10x = sc.read_10x_mtx(f'../../data/downstream/matrices/{sample}/10x/')
adata_final = sc.read_10x_mtx(f'../../data/downstream/matrices/{sample}/final/')

In [5]:
percentages = {}
diffs = {}
gen1 = {}
gen2 = {}

for sample in samples:
    # Loading cell-gene matrices:
    adata_10x = sc.read_10x_mtx(f'../../data/downstream/matrices/{sample}/10x/')
    adata_final = sc.read_10x_mtx(f'../../data/downstream/matrices/{sample}/final/')
    percentages[sample] = (adata_final.X.sum()-adata_10x.X.sum())/adata_10x.X.sum()*100
    
    ge1 = np.array(adata_10x.X.sum(axis=0)).flatten()
    ge2 = np.array(adata_final.X.sum(axis=0)).flatten()
    genes1 = adata_10x.var_names
    genes2 = adata_final.var_names
    gen1[sample] = genes1
    gen2[sample] = genes2
    expr1 = dict(zip(genes1, ge1))
    expr2 = dict(zip(genes2, ge2))
    all_genes = set(genes1) | set(genes2)
    diff = np.array([[g, int(expr2.get(g, 0) - expr1.get(g, 0)), int(expr1.get(g, 0)), int(expr2.get(g, 0))] for g in all_genes])
    diff = diff[diff[:, 1].astype(int).argsort()]
    diffs[sample] = diff

In [21]:
np.sum([x for x in diffs['brain'][:,1].astype(int)])

2241237

In [6]:
stats = {}
lost_genes = {}
new_genes = {}
for sample in samples:
    depth = diffs[sample][:,2].astype(int).sum()
    cpm = depth / 1000000
    lost = len([x for x in diffs[sample] if int(x[3]) < 0.1 * cpm and int(x[2]) > cpm])
    new  = len([x for x in diffs[sample] if int(x[2]) < 0.1 * cpm and int(x[3]) > cpm])
    lost_genes[sample] = [x[0] for x in diffs[sample] if int(x[3]) < 0.1 * cpm and int(x[2]) > cpm]
    new_genes[sample]  = [x[0] for x in diffs[sample] if int(x[2]) < 0.1 * cpm and int(x[3]) > cpm]
    added_genes = diffs[sample]
    stats[sample] = [lost, new]

In [9]:
print(np.sum([len(x) for key,x in lost_genes.items()]))
lost_set = set()
for key, x in lost_genes.items():
    lost_set.update(x)
print(len(lost_set))

183
62


In [10]:
print(np.sum([len(x) for key,x in new_genes.items()]))
new_set = set()
for key, x in new_genes.items():
    new_set.update(x)
print(len(new_set))

4897
1030


In [11]:
np.mean([x for key, x in percentages.items()])

1.2031416441979152

In [12]:
print(f"{'Sample':<15} {'Lost': <5} {'New': <5}")
for sample in samples:
    print(f"{sample:<15} {stats[sample][0]: <5} {stats[sample][1]: <5}")

Sample          Lost  New  
PBMC_10x        10    286  
PBMC_10x_2      11    412  
PBMC_10x_3      12    377  
PBMC_indrops    8     214  
PBMC_indrops_2  15    389  
brain           14    384  
brain_2         18    345  
eye             16    440  
eye_2           11    314  
eye_3           16    347  
lung_2          14    355  
lung_5          13    351  
lung_7          13    364  
lung_8          12    319  


In [13]:
print(np.mean([x[0] for key, x in stats.items()]))
print(np.mean([x[1] for key, x in stats.items()]))

13.071428571428571
349.7857142857143


In [14]:
for sample in samples:
    print(f"{sample:<15} {percentages[sample]}")

PBMC_10x        1.0363533161580563
PBMC_10x_2      0.9612013585865498
PBMC_10x_3      0.9441932663321495
PBMC_indrops    1.5338955447077751
PBMC_indrops_2  0.5590002052485943
brain           2.090439759194851
brain_2         1.7422476783394814
eye             1.3318334706127644
eye_2           1.5665752813220024
eye_3           1.543388795107603
lung_2          0.6658453959971666
lung_5          0.7305926643311977
lung_7          1.1411787010729313
lung_8          0.9972375817596912


In [10]:
print(diffs[sample][-10:][:,0])

['NBL1' 'MEMO1' 'SUMO2' 'STRADA' 'PTP4A2' 'CMAHP' 'ALDOA' 'HNRNPA1'
 'ANKRD44' 'TYMP']


In [15]:
print(diffs[sample][:10][:,0])

['TREM1' 'ENSG00000290677' 'RPL26' 'RPL21' 'RPL6' 'PSMA1' 'PDE4DIP'
 'SSBP1' 'RPL13A' 'MAN2B1']


In [22]:
ensg_ids = [
    "ENSG00000037637", "ENSG00000227751", "ENSG00000119535", "ENSG00000137959",
    "ENSG00000117228", "ENSG00000134183", "ENSG00000213064", "ENSG00000116741",
    "ENSG00000148841", "ENSG00000255949", "ENSG00000153179", "ENSG00000120860",
    "ENSG00000089234", "ENSG00000289979", "ENSG00000198252", "ENSG00000269910",
    "ENSG00000259617", "ENSG00000166710", "ENSG00000264937", "ENSG00000157890",
    "ENSG00000275454", "ENSG00000167721", "ENSG00000262884", "ENSG00000185245",
    "ENSG00000171962", "ENSG00000185862", "ENSG00000062716", "ENSG00000262413",
    "ENSG00000141429", "ENSG00000301067.1", "ENSG00000205784", "ENSG00000099783",
    "ENSG00000179271", "ENSG00000130522", "ENSG00000028277", "ENSG00000104804",
    "ENSG00000275183", "ENSG00000119801", "ENSG00000138081", "ENSG00000138071",
    "ENSG00000168894", "ENSG00000273306", "ENSG00000115232", "ENSG00000272807",
    "ENSG00000179921", "ENSG00000089012", "ENSG00000183597", "ENSG00000198951",
    "ENSG00000243410", "ENSG00000184897", "ENSG00000082074", "ENSG00000152348",
    "ENSG00000113719", "ENSG00000137393", "ENSG00000197061", "ENSG00000276903",
    "ENSG00000096060", "ENSG00000112576", "ENSG00000188820", "ENSG00000146425",
    "ENSG00000112531", "ENSG00000136279", "ENSG00000122678", "ENSG00000127951",
    "ENSG00000120910", "ENSG00000197265", "ENSG00000300552.1", "ENSG00000254087",
    "ENSG00000235298", "ENSG00000135049", "ENSG00000130713", "ENSG00000273748",
    "ENSG00000120280", "ENSG00000181704", "ENSG00000131171"
]

In [23]:
diffs[sample]

array([['TREM1', '-65481', '67028', '1547'],
       ['ENSG00000290677', '-59416', '62376', '2960'],
       ['RPL26', '-45564', '257135', '211571'],
       ...,
       ['TREM1-1', '67026', '0', '67026'],
       ['ALDOA', '81090', '0', '81090'],
       ['IFI6', '125075', '17120', '142195']], dtype='<U25')

In [25]:
ensg_to_gene = dict(zip(adata_10x.var['gene_ids'], adata_10x.var.index))

# Convert ENSG IDs to gene names
gene_names = [ensg_to_gene.get(ensg_id, 'Unknown') for ensg_id in ensg_ids]

In [26]:
extensions = [x for x in diffs[sample] if x[0] in gene_names]

In [28]:
np.sum([int(x[1]) for x in extensions])

24903

In [29]:
np.sum([int(x[1]) for x in diffs[sample]])

1411620

In [31]:
adata_10x.X.sum()

146862880.0

In [30]:
24903/1411620*100

1.7641433246907807

In [32]:
24903/146862880*100

0.016956633289501064

In [130]:
for i in range(len(extensions)):
    extensions[i] = np.append(extensions[i], float(extensions[i][2]) / float(extensions[i][3]))

In [132]:
extensions = sorted(extensions, key=lambda x: x[4])

In [133]:
extensions

[array(['ENSG00000264937', '299', '100', '399', '0.2506265664160401'],
       dtype='<U32'),
 array(['RCC2-AS1', '413', '179', '592', '0.30236486486486486'],
       dtype='<U32'),
 array(['ENSG00000262884', '245', '122', '367', '0.33242506811989103'],
       dtype='<U32'),
 array(['GNAT2', '249', '128', '377', '0.3395225464190981'], dtype='<U32'),
 array(['RPS6KB2-AS1', '506', '389', '895', '0.4346368715083799'],
       dtype='<U32'),
 array(['ENSG00000273306', '75', '87', '162', '0.5370370370370371'],
       dtype='<U32'),
 array(['GP1BA', '238', '291', '529', '0.5500945179584121'], dtype='<U32'),
 array(['ENSG00000289979', '213', '266', '479', '0.5553235908141962'],
       dtype='<U32'),
 array(['ENSG00000262413', '90', '132', '222', '0.5945945945945946'],
       dtype='<U32'),
 array(['HNRNPK-AS1', '875', '1639', '2514', '0.6519490851233095'],
       dtype='<U32'),
 array(['MEGF11', '182', '356', '538', '0.6617100371747212'], dtype='<U32'),
 array(['DRC3', '334', '679', '1013', '0.6

In [120]:
adata_10x.var

,gene_ids,feature_types
DDX11L2,ENSG00000290825,Gene Expression
MIR1302-2HG,ENSG00000243485,Gene Expression
FAM138A,ENSG00000237613,Gene Expression
ENSG00000290826,ENSG00000290826,Gene Expression
OR4F5,ENSG00000186092,Gene Expression
...,...,...
ENSG00000277836,ENSG00000277836,Gene Expression
ENSG00000278633,ENSG00000278633,Gene Expression
ENSG00000276017,ENSG00000276017,Gene Expression
ENSG00000278817,ENSG00000278817,Gene Expression


In [13]:
gene_list = ['GNLY', 'KLRD1', 'KLRF1', 'CTSW', 'XCL2', 'NCALD', 'SAMD3', 'HOPX', 'IL2RB', 'NCAM1', 'CMC1', 'TRDC', 'LINC00299', 'ID2', 'ATP8B4', 'NKG7', 'KLRC2', 'CD7', 'XCL1', 'PPP1R9A', 'TXK', 'KLRB1', 'IL18RAP', 'UST', 'PRF1']

In [16]:
for x in diffs[sample]:
    if x[0] in gene_list:
        print(x)

['KLRD1' '-4' '7574' '7570']
['XCL1' '-3' '328' '325']
['PRF1' '-1' '9050' '9049']
['ATP8B4' '-1' '10465' '10464']
['NCALD' '-1' '10952' '10951']
['ID2' '0' '15266' '15266']
['CD7' '0' '13510' '13510']
['GNLY' '0' '59523' '59523']
['NCAM1' '0' '2256' '2256']
['TXK' '0' '27133' '27133']
['PPP1R9A' '0' '1048' '1048']
['CTSW' '0' '11787' '11787']
['KLRF1' '0' '4452' '4452']
['UST' '0' '7851' '7851']
['XCL2' '0' '1613' '1613']
['LINC00299' '0' '2457' '2457']
['HOPX' '0' '5904' '5904']
['TRDC' '0' '3499' '3499']
['CMC1' '0' '12495' '12495']
['KLRC2' '0' '1433' '1433']
['IL2RB' '0' '3082' '3082']
['NKG7' '0' '39112' '39112']
['KLRB1' '0' '10287' '10287']
['IL18RAP' '273' '774' '1047']
['SAMD3' '1226' '14834' '16060']


['GNLY' '0' '59523' '59523']
